In [1]:
from datasets import load_dataset
import pandas as pd
import re, json

pd.set_option("display.max_rows", 300)

DATASET = "lang-uk/recruitment-dataset-job-descriptions-english"
df = load_dataset(DATASET, split="train").to_pandas()
print(f"{len(df):,} rows, {df.shape[1]} columns")
df.head(3)

/opt/anaconda3/envs/resumechecker/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


141,897 rows, 10 columns


,Position,Long Description,Company Name,Exp Years,Primary Keyword,English Level,Published,Long Description_lang,id,__index_level_0__
0,10 + Blockchain Nodes / Masternodes to set up,*Requirements*\r\n\r\nWe're looking for a long...,MyCointainer,2y,Sysadmin,intermediate,2020-10-01T00:00:00+03:00,en,c0ca96e7-85df-50df-a64e-d934cd02a170,27461
1,10 .NET Developers (Middle and Senior level),"Greetings! My name is Maria, I am in urgent ne...",TechScout.tech,2y,.NET,intermediate,2022-03-01T00:00:00+02:00,en,64f4b7ea-36e4-5bdd-a8b1-185f32f7dc7f,27462
2,"10X Engineer (co-founder, #4 employee, USD 11-...",**Product**\r\nThe product is a live video cha...,Innoteka,5y,JavaScript,fluent,2021-07-01T00:00:00+03:00,en,b9a1303e-dd0c-5ed1-8f62-be2bc4c7da4f,27463


In [2]:
print(df.columns.tolist())
df.info()

['Position', 'Long Description', 'Company Name', 'Exp Years', 'Primary Keyword', 'English Level', 'Published', 'Long Description_lang', 'id', '__index_level_0__']
<class 'pandas.DataFrame'>
RangeIndex: 141897 entries, 0 to 141896
Data columns (total 10 columns):
 #   Column                 Non-Null Count   Dtype
---  ------                 --------------   -----
 0   Position               141897 non-null  str  
 1   Long Description       141897 non-null  str  
 2   Company Name           141897 non-null  str  
 3   Exp Years              141897 non-null  str  
 4   Primary Keyword        141897 non-null  str  
 5   English Level          134358 non-null  str  
 6   Published              141897 non-null  str  
 7   Long Description_lang  141897 non-null  str  
 8   id                     141897 non-null  str  
 9   __index_level_0__      141897 non-null  int64
dtypes: int64(1), str(9)
memory usage: 273.1 MB


In [3]:
pk = df["Primary Keyword"].value_counts(dropna=False)
print(f"{pk.shape[0]} distinct Primary Keyword values\n")
pk

45 distinct Primary Keyword values



Primary Keyword
JavaScript           17903
Java                  8712
DevOps                7979
.NET                  7826
Other                 7536
QA Automation         7047
Marketing             6933
QA                    6774
Node.js               6416
PHP                   5740
Python                5735
Project Manager       4618
HR                    4063
Design                4020
Sales                 4014
C++                   3743
Business Analyst      3679
Support               2819
Android               2651
iOS                   2398
Data Science          2262
Product Manager       1939
Lead                  1861
Ruby                  1834
Golang                1747
Sysadmin              1694
SQL                   1459
Data Engineer         1451
Unity                 1165
Lead Generation        840
Security               830
Data Analyst           767
Recruiter              604
Artist                 554
Scala                  492
Technical Writing      426
SEO         

In [4]:
print("TECH_KEYWORDS = {")
for kw in sorted(df["Primary Keyword"].dropna().unique()):
    print(f"    {kw!r},")
print("}")

TECH_KEYWORDS = {
    '.NET',
    'Android',
    'Artist',
    'Block-chain',
    'Business Analyst',
    'C++',
    'Data Analyst',
    'Data Engineer',
    'Data Science',
    'Design',
    'DevOps',
    'Flutter',
    'Golang',
    'HR',
    'Java',
    'JavaScript',
    'Lead',
    'Lead Generation',
    'Marketing',
    'Node.js',
    'Other',
    'PHP',
    'Product Manager',
    'Product Owner',
    'Project Manager',
    'Python',
    'QA',
    'QA Automation',
    'React',
    'Recruiter',
    'Ruby',
    'Rust',
    'SAP',
    'SEO',
    'SQL',
    'Sales',
    'Salesforce',
    'Scala',
    'Scrum Master',
    'Security',
    'Support',
    'Sysadmin',
    'Technical Writing',
    'Unity',
    'iOS',
}


In [5]:
df["Exp Years"].value_counts(dropna=False)

Exp Years
3y        49996
2y        36150
5y        27772
1y        22007
no_exp     5972
Name: count, dtype: int64

In [6]:
# >>> EDIT ME <<<  seed list — prune/extend using the value_counts above.
# Use the EXACT strings as they appear (case-sensitive).
TECH_KEYWORDS = {
    '.NET', 'Android', 'Block-chain', 'C++', 'Data Analyst', 'Data Engineer',
    'Data Science', 'DevOps', 'Flutter', 'Golang', 'Java', 'JavaScript',
    'Node.js', 'PHP', 'Python', 'QA', 'QA Automation', 'React', 'Ruby',
    'Rust', 'Scala', 'Security', 'SQL', 'Sysadmin', 'Unity', 'iOS', 'SAP',
    'Salesforce', 'Business Analyst'
}

In [7]:
# 1) tech roles — exact match on Primary Keyword
tech_mask = df["Primary Keyword"].isin(TECH_KEYWORDS)

# 2) experience 1-5 years inclusive. 'Exp Years' is a string -> pull the first number.
exp_years = df["Exp Years"].str.extract(r"(\d+(?:\.\d+)?)", expand=False).astype(float)
exp_mask = exp_years.between(1, 5, inclusive="both")

sub = df[tech_mask & exp_mask].copy()
sub["exp_years"] = exp_years[tech_mask & exp_mask]
print(f"tech: {tech_mask.sum():,} | 1-5y: {exp_mask.sum():,} | both: {len(sub):,}")
sub["Primary Keyword"].value_counts().head(20)

tech: 101,015 | 1-5y: 135,925 | both: 98,070


Primary Keyword
JavaScript          17486
Java                 8504
DevOps               7807
.NET                 7574
QA Automation        6840
Node.js              6301
QA                   6264
PHP                  5621
Python               5616
C++                  3580
Business Analyst     3566
Android              2579
iOS                  2339
Data Science         2181
Ruby                 1804
Golang               1709
Sysadmin             1626
Data Engineer        1425
SQL                  1409
Unity                1131
Name: count, dtype: int64

In [8]:
BULLET = re.compile(r"^\s*[•●▪‣◦\-\*·–]+\s*")
REQ_HDR = re.compile(
    r"(?i)\b(requirements?|required skills?|must[- ]?have|qualifications?|"
    r"what (we expect|you('?ll| will)? need)|you (should )?have|"
    r"necessary skills?|hard skills?|skills? (&|and) experience)\b")
PREF_HDR = re.compile(
    r"(?i)\b(nice[- ]?to[- ]?have|preferred|would be a plus|will be a plus|"
    r"good to have|as a plus|bonus|optional|desirable|advantage)\b")
OTHER_HDR = re.compile(
    r"\b(about|who we are|company|team|benefits|perks|what we offer|"
    r"responsibilities|role|overview|description|culture|mission)\b",
    re.IGNORECASE,
)

def parse_description(text):
    mins, prefs, bucket = [], [], None
    rest = []
    for raw in (text or "").splitlines():
        line = raw.strip()
        if not line:
            continue
        short = len(line.split()) <= 8          # headers are short lines
        if short and PREF_HDR.search(line):
            bucket = "pref"; continue
        if short and REQ_HDR.search(line):
            bucket = "min"; continue
        if short and OTHER_HDR.search(line):
            bucket = None; continue
        clean = BULLET.sub("", line).strip()
        if not clean:
            continue
        if bucket == "min":
            mins.append(clean)
        elif bucket == "pref":
            prefs.append(clean)
        else:
            rest.append(clean)
    return mins, prefs, rest

In [ ]:
parsed = sub["Long Description"].apply(parse_description)
sub["minimum_requirements"]     = parsed.apply(lambda t: t[0])
sub["preferred_qualifications"] = parsed.apply(lambda t: t[1])
sub["other_information"]          = parsed.apply(lambda t: t[2])


out = (sub.rename(columns={"id": "job_id",
                           "Position": "job_title",
                           "Primary Keyword": "primary_keyword",
                           "Long Description": "description"})
          [["job_id", "job_title", "primary_keyword", "exp_years",
            "minimum_requirements", "preferred_qualifications", "other_information", "description"]]
          .drop_duplicates(subset="job_id")
          .set_index("job_id"))
print(f"{len(out):,} unique tech jobs")a
out.head(3)

98,070 unique tech jobs


,job_title,primary_keyword,exp_years,minimum_requirements,preferred_qualifications,other_information,description
job_id,,,,,,,
c0ca96e7-85df-50df-a64e-d934cd02a170,10 + Blockchain Nodes / Masternodes to set up,Sysadmin,2.0,[We're looking for a long term collaboration w...,[],[],*Requirements*\r\n\r\nWe're looking for a long...
64f4b7ea-36e4-5bdd-a8b1-185f32f7dc7f,10 .NET Developers (Middle and Senior level),.NET,2.0,[2+ years of hands-on experience of .Net devel...,[],"[Greetings! My name is Maria, I am in urgent n...","Greetings! My name is Maria, I am in urgent ne..."
b9a1303e-dd0c-5ed1-8f62-be2bc4c7da4f,"10X Engineer (co-founder, #4 employee, USD 11-...",JavaScript,5.0,[],[],"[Product**, The product is a live video chat a...",**Product**\r\nThe product is a live video cha...


In [10]:
out.to_json("job_descriptions_tech.json", orient="index", force_ascii=False, indent=2)

csv_df = out.copy()
for c in ("minimum_requirements", "preferred_qualifications"):
    csv_df[c] = csv_df[c].apply(lambda v: json.dumps(v, ensure_ascii=False))
csv_df.to_csv("job_descriptions_tech.csv")
print("wrote job_descriptions_tech.json and job_descriptions_tech.csv")

wrote job_descriptions_tech.json and job_descriptions_tech.csv


In [11]:
d = pd.read_csv('job_descriptions_tech.csv')

# keep rows whose preferred_qualifications list isn't empty
d = d[(d['preferred_qualifications'] != '[]') & (d['minimum_requirements'] != '[]')].copy()
print(f"{len(d):,} jobs with non-empty preferred_qualifications")

d.to_csv('job_descriptions_tech_pref.csv', index=False)
print("wrote job_descriptions_tech_pref.csv")

33,065 jobs with non-empty preferred_qualifications
wrote job_descriptions_tech_pref.csv
